# Laboratorio 3 — Preparación de los datos

**Asignatura:** Analítica de Datos  
**Caso:** Andina Retail

## Objetivo

Aplicar transformaciones justificadas a los problemas identificados durante la evaluación de calidad de los datos.

En este laboratorio seguiremos el proceso:

**problema detectado → decisión → transformación → verificación**

Trabajaremos nuevamente desde el archivo original `ventas.csv` para mantener un proceso reproducible.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
df = pd.read_csv("/content/ventas.csv")
df.shape

(4820, 10)

## 1. Corrección del tipo de fecha

Durante la inspección inicial observamos que `fecha` fue interpretada por Pandas como `object`.

Sin embargo, su significado analítico es temporal.

### Antes de transformar

Primero verificaremos el tipo detectado actualmente.

In [4]:
df["fecha"].dtype

dtype('O')

In [5]:
type(df["fecha"][0])

str

### Transformación
Convertiremos la columna utilizando `pd.to_datetime()` para que Pandas pueda tratar sus valores como fechas.

In [6]:
df["fecha"] = pd.to_datetime(df["fecha"])

### Verificación

Comprobamos nuevamente el tipo de la variable.

In [7]:
df["fecha"].dtype

dtype('<M8[ns]')

In [8]:
type(df["fecha"][0])

pandas._libs.tslibs.timestamps.Timestamp

In [9]:
df["fecha"][0]

Timestamp('2025-11-03 00:00:00')

## 2. Normalización de categorías

En el Laboratorio 2 encontramos diferentes representaciones para una misma ciudad:

- `Bogota` y `bogota`
- `Medellin` y `MEDELLIN`

Para Pandas son categorías diferentes, aunque analíticamente representan la misma ciudad.

### Antes de transformar

Observemos nuevamente las categorías existentes.

In [10]:
df["ciudad"].value_counts()

,count
ciudad,
Medellin,1402
Bogota,1310
Cali,1080
Barranquilla,900
MEDELLIN,87
bogota,41


### Transformación

Aplicaremos tres operaciones:

1. `str.strip()` elimina espacios innecesarios al inicio y al final.
2. `str.lower()` convierte el texto a minúsculas.
3. `str.title()` estandariza la escritura utilizando inicial mayúscula.

El objetivo es obtener una representación consistente de cada ciudad.

In [11]:
type(df["ciudad"][0])

str

In [12]:
df["ciudad"] = (
    df["ciudad"]
    .str.strip()
    .str.lower()
    .str.title()
)

### Verificación

Después de la transformación debemos comprobar nuevamente las categorías y su cardinalidad.

In [13]:
df["ciudad"].value_counts()

,count
ciudad,
Medellin,1489
Bogota,1351
Cali,1080
Barranquilla,900


In [14]:
df["ciudad"].nunique()

4

## 3. Tratamiento de valores faltantes en `descuento`

En el Laboratorio 2 encontramos:

- 118 valores faltantes en `descuento`.
- Estos representan aproximadamente el 2.45 % de las transacciones.

Un valor faltante no debe reemplazarse automáticamente.

### Decisión para este ejercicio

Para este dataset académico estableceremos la siguiente regla:

> Un descuento faltante será interpretado como una transacción sin descuento registrado y se representará mediante `0`.

Esta decisión debe quedar documentada porque modifica los datos originales.

In [19]:
df["descuento"].isnull().sum()

np.int64(0)

### Transformación

Utilizaremos `fillna(0)` para reemplazar únicamente los valores faltantes de `descuento`.

In [18]:
df["descuento"] = df["descuento"].fillna(0)

### Verificación

In [20]:
df["descuento"].isnull().sum()

np.int64(0)

## 4. Reconstrucción de valores faltantes en `ventas`

En el Laboratorio 2 identificamos 24 transacciones sin valor registrado en `ventas`.

En este dataset, `ventas` representa el **valor monetario final de la transacción**.

Disponemos de las variables necesarias para calcularlo:

- `unidades`
- `precio_unitario`
- `descuento`

La relación utilizada en el dataset es:
$$
\text{ventas} = \text{unidades} \times \text{precio_unitario} \times (1-\text{descuento})
$$

Por tanto, reconstruiremos únicamente los valores faltantes utilizando la información disponible en cada transacción.

In [21]:
df["descuento"].unique()

array([0.05, 0.  , 0.1 , 0.15, 0.2 , 0.3 ])

In [22]:
df["ventas"].isnull().sum()

np.int64(24)

### Identificación de las transacciones afectadas

Creamos una condición que identifica únicamente las filas donde `ventas` está ausente.

In [23]:
faltan_ventas = df["ventas"].isnull()
df.loc[
    faltan_ventas,
    ["unidades", "precio_unitario", "descuento", "ventas"]
]

,unidades,precio_unitario,descuento,ventas
11,1,66200.0,0.10,NaN
854,3,110200.0,0.00,NaN
945,4,107700.0,0.05,NaN
1109,4,70500.0,0.00,NaN
1209,3,102900.0,0.00,NaN
1807,2,584600.0,0.00,NaN
1902,2,168300.0,0.05,NaN
1975,3,72300.0,0.05,NaN
2195,3,29300.0,0.05,NaN
2304,4,78300.0,0.05,NaN


### Imputación de valores faltantes

En esta etapa **imputaremos los valores faltantes de la columna `ventas`**. Para ello, no eliminaremos las transacciones que tienen `ventas` ausente, sino que **calcularemos su valor a partir de las demás variables disponibles**.

La fórmula utilizada es:

**`ventas = unidades × precio_unitario × (1 − descuento)`**

El cálculo se aplicará **únicamente a las filas donde `ventas` está ausente (`NaN`)**, identificadas mediante `faltan_ventas`. De esta manera, los valores de `ventas` que ya existen no se modifican.

```python
df.loc[faltan_ventas, "ventas"] = (
    df.loc[faltan_ventas, "unidades"]
    * df.loc[faltan_ventas, "precio_unitario"]
    * (1 - df.loc[faltan_ventas, "descuento"])
)

In [24]:
df.loc[faltan_ventas, "ventas"] = (
    df.loc[faltan_ventas, "unidades"]
    * df.loc[faltan_ventas, "precio_unitario"]
    * (1 - df.loc[faltan_ventas, "descuento"])
)

### Verificación

Comprobamos que ya no existan valores faltantes en `ventas`.

In [28]:
df.loc[df["ventas"].isnull()]#buscar las filas donde están los datos faltantes

,fecha,id_cliente,producto,categoria,ciudad,unidades,precio_unitario,descuento,canal,ventas


In [27]:
df["ventas"]

,ventas
0,182780.0
1,409000.0
2,135360.0
3,126900.0
4,89775.0
...,...
4815,122700.0
4816,78080.0
4817,132700.0
4818,425600.0


## 5. ¿Debemos eliminar duplicados?

Durante el Laboratorio 2 verificamos la existencia de filas exactamente duplicadas.

Antes de aplicar cualquier transformación debemos comprobar nuevamente el resultado.

In [29]:
df.duplicated().sum()

np.int64(0)

### Decisión

No se detectaron registros exactamente duplicados.

Por tanto, **no realizaremos ninguna eliminación**.

Este resultado muestra que preparar datos no consiste en ejecutar automáticamente operaciones como `drop_duplicates()`.

Una transformación solo debe aplicarse cuando existe un problema identificado y una razón que la justifique.

## 6. Revisión del valor máximo de `unidades`

Durante el perfilado encontramos una transacción con un valor de `unidades` considerablemente superior al resto.

Antes de eliminar o modificar un valor inusual debemos inspeccionar la observación correspondiente.

In [30]:
df.loc[df["unidades"] == df["unidades"].max()]

,fecha,id_cliente,producto,categoria,ciudad,unidades,precio_unitario,descuento,canal,ventas
1700,2025-12-25,14901,Portatil 14,Tecnologia,Medellin,500,3200000.0,0.0,Tienda,12450000.0


### Decisión

El hecho de que un valor sea inusual no demuestra que sea incorrecto.

No contamos con evidencia suficiente para afirmar que la transacción con el máximo número de unidades sea un error.

Por tanto:

**el registro se conserva y se documenta para un análisis posterior.**

En el análisis exploratorio estudiaremos con mayor detalle la distribución de esta variable y los posibles valores atípicos.

# 7. Verificación final de la preparación

Después de realizar las transformaciones debemos volver a evaluar el dataset.

La preparación no termina cuando ejecutamos el código: termina cuando comprobamos que las transformaciones produjeron el resultado esperado.

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4820 entries, 0 to 4819
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   fecha            4820 non-null   datetime64[ns]
 1   id_cliente       4820 non-null   int64         
 2   producto         4820 non-null   object        
 3   categoria        4820 non-null   object        
 4   ciudad           4820 non-null   object        
 5   unidades         4820 non-null   int64         
 6   precio_unitario  4820 non-null   float64       
 7   descuento        4820 non-null   float64       
 8   canal            4820 non-null   object        
 9   ventas           4820 non-null   float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(4)
memory usage: 376.7+ KB


In [32]:
df.isnull().sum()

,0
fecha,0
id_cliente,0
producto,0
categoria,0
ciudad,0
unidades,0
precio_unitario,0
descuento,0
canal,0
ventas,0


In [33]:
df["ciudad"].value_counts()

,count
ciudad,
Medellin,1489
Bogota,1351
Cali,1080
Barranquilla,900


## Resumen del estado final

Comprobaremos algunos indicadores básicos del dataset después de la preparación.

In [34]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
print("Valores faltantes:", df.isnull().sum().sum())
print("Duplicados:", df.duplicated().sum())
print("Ciudades:", df["ciudad"].nunique())
print("Tipo de fecha:", df["fecha"].dtype)

Filas: 4820
Columnas: 10
Valores faltantes: 0
Duplicados: 0
Ciudades: 4
Tipo de fecha: datetime64[ns]


# 8. Generación del dataset preparado

No sobrescribiremos el archivo original.

Conservaremos:

- `ventas.csv` → datos originales;
- `ventas_preparadas.csv` → datos después del proceso de preparación.

Esto permite conservar la trazabilidad del proceso y reproducir las transformaciones realizadas.

In [35]:
df.to_csv("ventas_preparadas.csv", index=False)

El archivo `ventas_preparadas.csv` será utilizado como punto de partida para el análisis exploratorio.

# 9. Conclusiones

Durante este laboratorio:

- convertimos `fecha` a un tipo temporal;
- normalizamos las categorías de `ciudad`;
- tratamos los valores faltantes de `descuento` mediante una regla documentada;
- reconstruimos los valores faltantes de `ventas` utilizando la relación entre las variables disponibles;
- comprobamos que no existían duplicados exactos que eliminar;
- conservamos el valor inusual de `unidades` porque no existe evidencia suficiente para considerarlo un error;
- verificamos nuevamente la calidad del dataset;
- generamos un nuevo archivo sin modificar los datos originales.

## Idea central

La preparación de datos no consiste simplemente en ejecutar comandos de limpieza.

Cada transformación debe seguir el proceso:

**problema → decisión justificada → transformación → verificación**

---

## Pregunta para continuar

Ahora que disponemos de datos preparados:

**¿Qué patrones, distribuciones y diferencias podemos encontrar en las transacciones de Andina Retail?**

Esta pregunta nos conduce al:

### Laboratorio 4 — Análisis Exploratorio de Datos (EDA)